# ai_water: train TerraMind on JRC-labelled Sentinel-1 water maps

Trains the water model on **six water bodies** (`data/training_sites.csv`: Angat, Magat and La Mesa dams, Sampaloc Lake, Libmanan and Cagayan river
reaches), on real data, end to end:

1. **Dataset** - Sentinel-1 RTC (Planetary Computer, **ascending and descending** passes) -> refined Lee -> VV, VH and **SDWI**; the site's **DEM** (slope, aspect,
   relief) and **land cover** (ESA WorldCover, context only) from Earth Engine. Labels are the **JRC Monthly History map of the same calendar month** (Earth Engine),
   corrected with the terrain: JRC land that is dark in the radar (shadow, wet soil) becomes an explicit *hard negative*, and JRC water on a steep slope is ignored.
   There is no Otsu thresholding anywhere. Scenes are written to Drive one by one, so an interrupted run **resumes**.
2. **Training** - pretrained TerraMind (`S1RTC` modality, Hugging Face) reads VV and VH; a light decoder also sees SDWI, terrain and land cover. Land cover is blanked
   in half of the training samples so it stays a hint, never the decider. One or more whole sites are held out to measure how well the model carries over.
3. **Saving** - a full checkpoint (Drive), a small one for GitHub (`models/`), a model card.

Then run **`ai_water_inference.ipynb`** to map any water body (Angat Dam by default), many bodies, or the whole country.

Before you start: an Earth Engine account with a Google Cloud project (set `GEE_PROJECT`), Google Drive, and a **GPU runtime**. Dataset building is the long
step (about 125 passes per site and direction for 2018-2021, network-bound). Labelled dates are 2017-2021 (Sentinel-1 RTC starts in 2017, JRC GSW1_4 ends in
Dec 2021); the trained model then runs on any date.

## 0. Setup

In [ ]:
import os, sys, json, subprocess, tempfile
from pathlib import Path

# Colab: install the dependencies. numpy is pinned below 2.3 because Colab's preinstalled numba requires it; without the pin pip
# upgrades numpy and prints "numba requires numpy<2.3" (and numba then fails to import). If numpy was already imported in this
# session, restart the runtime once after this cell (Runtime > Restart session) and run the notebook again.
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2.3", "scipy", "scikit-image", "shapely", "pandas",
                    "matplotlib", "rasterio", "affine", "pyproj", "pystac-client", "planetary-computer", "odc-stac",
                    "earthengine-api", "torch", "terratorch", "geopandas"], check=True)

# repository root: hard-coded (edit this one line if the project lives elsewhere)
ROOT = Path("/content/drive/MyDrive/Surface_Water/project")
if "google.colab" in sys.modules and not ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")
assert (ROOT / "ai_water").is_dir(), f"ai_water/ not found in {ROOT}: check the path above and that Drive is mounted"
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

## 1. Configuration

In [ ]:
# ---- training sites and labels
TRAIN_SITES_FILE = ROOT / "data" / "training_sites.csv"     # 3 dams, 1 lake, 2 river reaches (edit the file to change them)
HOLDOUT = ["La Mesa Dam"]           # whole sites kept out of training and used for validation ([] = east-strip block split)
START, END = "2018-01-01", "2021-12-31"     # Sentinel-1 dates to label (JRC GSW1_4 ends in Dec 2021)
GEE_PROJECT = None                  # <-- your Google Cloud project id for Earth Engine
MAX_MONTH_OFFSET = 0                # 0 = the JRC month must be the Sentinel-1 month (recommended)
MIN_CONTRAST_DB = 5.0               # scenes where JRC water is less than this much darker (VV, dB) than JRC land are rejected
MAX_WATER_SLOPE = 15.0              # JRC water on a slope steeper than this (degrees) is ignored: open water is flat
DARK_DB = -18.0                     # JRC land darker than this in the radar = hard negative (shadow, wet soil)
USE_LANDCOVER = True                # ESA WorldCover groups as context channels (never the decider: blanked in half of the training samples)
RESTART = False                     # True: delete the manifest and rebuild the dataset from scratch (False: resume)

# ---- model
BACKBONE = "terramind_v1_base"      # tiny | small | base | large
PRETRAINED = True                   # downloads ibm-esa-geospatial/TerraMind-1.0-* from Hugging Face
UNFREEZE_LAST = 0                   # 0 = train the decoder only; N > 0 also fine-tunes the last N transformer blocks
PATCH, EPOCHS, BATCH, LR = 224, 15, 8, 3e-4
KEEP_EMPTY = 0.2                    # share of near-waterless patches kept (water is rare; rivers especially)
MAX_PATCHES_PER_SCENE = 32          # patches are held in RAM (float16): ~0.2 MB each; raise it only if you have the memory
RES_M = 10.0

OUT = ROOT / "outputs"              # on Drive: survives a disconnected session
DATA, MODEL_DIR = OUT / "dataset", OUT / "model"
for d in (DATA, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from ai_water import (acquire, dataset, features, infer, jrc, labels as lb, sites, terrain,
                      terramind_seg as tm)

DEVICE = "cuda" if tm.torch.cuda.is_available() else "cpu"
print("torch", tm.torch.__version__, "| device:", DEVICE)
if DEVICE == "cpu" and not BACKBONE.endswith(("tiny", "small")):
    print("WARNING: no GPU. Training / inference with this backbone on CPU will be extremely slow: Runtime > Change runtime type > GPU.")
print("model input channels:", features.CHANNELS)

LAB_CMAP = ListedColormap(["#c9b79c", "#2b7bba", "#dddddd"])                     # land, water, ignored
def lab_img(l):
    return np.where(l == features.IGNORE, 2, l)

## 2. The training sites

Read from `data/training_sites.csv` (corner coordinates as `(lat, lon)` pairs). Boxes smaller than 5 km on a side are expanded about their centre, because the
model looks at 2.2 km patches and needs context (Sampaloc Lake is only 1.4 km across). Each site gets the UTM zone of its own location.

In [ ]:
TRAIN_SITES = sites.load_training_sites(TRAIN_SITES_FILE)          # registers them, so sites.get("magat dam") works
SITE_IDS = [s.name for s in TRAIN_SITES]
display(pd.DataFrame([{"site": s.label, "id": s.name, "kind": s.kind, "width_km": round(s.size_km()[0], 1), "height_km": round(s.size_km()[1], 1),
                       "crs": s.crs, "note": s.note, "held out": s.name in {sites.slug(h) for h in HOLDOUT}} for s in TRAIN_SITES]))
unknown = [h for h in HOLDOUT if sites.slug(h) not in SITE_IDS]
assert not unknown, f"HOLDOUT names not among the training sites: {unknown}; they are {SITE_IDS}"

## 3. Dataset: Sentinel-1 + JRC of the same month + terrain

For each Sentinel-1 date (every ascending and descending pass over the site): speckle-filtered VV, VH and SDWI; the JRC monthly map of the same calendar month on the same
10 m grid; the site's terrain (fetched once and cached); and the labels (`labels.jrc_labels`): JRC water = 1, JRC land = 0, with

- **hard negatives**: JRC land that is dark in this pass (VV below `DARK_DB`) is labelled land and counted - this is what teaches the model that radar shadow is not water;
- JRC water on a slope steeper than `MAX_WATER_SLOPE` is ignored (a JRC error, such as Landsat topographic shadow);
- a 30 m band around the JRC shoreline is ignored (the model sharpens it from the 10 m radar).

Scenes with too little JRC coverage, without a water body, or where JRC water is not at least `MIN_CONTRAST_DB` darker than JRC land (the JRC month does not fit the pass)
are rejected and listed in `manifest.jsonl` with the reason.

**Resume:** every scene is appended to `manifest.jsonl` as soon as it is done. Re-running this cell (after a disconnect, or with a longer `END`) skips the
passes already in the manifest without reading them again. Set `RESTART = True` to start over (also needed after changing `MIN_CONTRAST_DB` or other
labelling settings, which apply only to scenes processed from then on).

In [ ]:
cfg = dataset.DatasetConfig(out_dir=str(DATA), start=START, end=END, max_month_offset=MAX_MONTH_OFFSET, min_contrast_db=MIN_CONTRAST_DB,
                            max_water_slope=MAX_WATER_SLOPE, dark_db=DARK_DB, use_landcover=USE_LANDCOVER, cache_dir=str(OUT / "cache"))
if RESTART and (DATA / "manifest.jsonl").exists():
    (DATA / "manifest.jsonl").unlink()
    print("RESTART: manifest deleted, rebuilding from scratch")

try:
    jrc.init_gee(GEE_PROJECT)
except Exception as e:                                  # not authenticated in this session yet: opens the Google sign-in once
    print("Earth Engine init failed (", e.__class__.__name__, "), authenticating ...")
    jrc.init_gee(GEE_PROJECT, authenticate=True)
source, terrain_source = jrc.GeeJrc(), terrain.GeeTerrain()

GRIDS = {}
for site in TRAIN_SITES:
    name = site.name
    done = dataset.done_scenes(str(DATA), name)
    if done:
        print(f"[{name}] resuming: {len(done)} scene(s) already in the manifest")
    grid, scenes = dataset.iter_s1(site, cfg, skip_days=done)
    GRIDS[name] = grid
    dataset.build_site(site, cfg, scenes, grid, source, terrain_source)

In [ ]:
man = dataset.read_manifest(str(DATA)) if (DATA / "manifest.jsonl").exists() else []
tbl = dataset.manifest_table(str(DATA), RES_M)     # raises a RuntimeError saying why if the manifest is empty / nothing is usable
display(tbl.round(3))
print(tbl.groupby(["site", "orbit"])["usable"].agg(["sum", "count"]).rename(columns={"sum": "usable", "count": "scenes"}))
print("rejection reasons:", tbl["reject_reason"].value_counts().to_dict())

Look at one labelled scene. Grey in the label panel is **ignored** (JRC no-data, the shoreline band, JRC water on steep ground); the `source` panel shows who decided each pixel:
`jrc_water`, `jrc_land`, `hard_negative` (JRC land that is dark in the radar - the shadow the model has to learn to reject) and `terrain_ignored`.
The bottom row is what the model sees besides VV and VH: SDWI, slope, relief and the radar-look slope (positive = faces away from the sensor).

In [ ]:
m0 = next(m for m in man if m["usable"])
z = np.load(DATA / m0["site"] / f"{m0['date']}_{m0['orbit']}.npz")
terr = features.Terrain.load(DATA / m0["site"] / "terrain.npz")
mm = jrc.matched_month(source, GRIDS[m0["site"]], m0["date"], MAX_MONTH_OFFSET)
rs = features.range_slope(terr.slope, terr.aspect, m0["orbit"])
fig, ax = plt.subplots(2, 4, figsize=(18, 9))
ax[0, 0].imshow(z["vv_db"], cmap="gray", vmin=-25, vmax=0);                     ax[0, 0].set_title(f"VV dB (refined Lee)  {m0['site']} {m0['date']} {m0['orbit']}")
ax[0, 1].imshow(mm.water_class, cmap=ListedColormap(["#222", "#c9b79c", "#2b7bba"]), vmin=0, vmax=2, interpolation="nearest")
ax[0, 1].set_title(f"JRC {mm.month:%Y-%m} (offset {mm.offset_months:+d})")
ax[0, 2].imshow(lab_img(z["labels"]), cmap=LAB_CMAP, vmin=0, vmax=2, interpolation="nearest")
ax[0, 2].set_title(f"labels (JRC-water vs JRC-land contrast {m0['labels']['contrast_db']:.1f} dB)")
ax[0, 3].imshow(z["source"], cmap="viridis", vmin=0, vmax=4, interpolation="nearest")
ax[0, 3].set_title("source: 1 jrc water 2 jrc land 3 hard negative 4 steep")
ax[1, 0].imshow(z["sdwi"], cmap="magma");                                        ax[1, 0].set_title("SDWI = ln(10 VV VH) - 8")
ax[1, 1].imshow(terr.slope, cmap="terrain", vmin=0, vmax=45);                    ax[1, 1].set_title("slope (deg)")
ax[1, 2].imshow(terr.relief, cmap="terrain");                                    ax[1, 2].set_title("relief: height above the local low (m)")
ax[1, 3].imshow(rs, cmap="RdBu", vmin=-45, vmax=45);                             ax[1, 3].set_title(f"range slope, {m0['orbit']} (red = faces away)")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()
print("label sources (px):", {k: v for k, v in m0["labels"].items() if k.endswith("_px")})

## 4. Patches and split

Patches with almost no water are subsampled (`KEEP_EMPTY`) so the rare class is not drowned, and at most `MAX_PATCHES_PER_SCENE` patches are kept per scene
(float16; only the three S1 channels are stored, the terrain channels are cut from the site raster when a batch is built) so everything fits in RAM.
Patches that contain hard negatives are kept even when they hold no water. With `HOLDOUT` sites the validation set is whole sites the model never saw - the honest test of generalisation (for example train on five bodies, test on La Mesa Dam).

In [ ]:
ps = dataset.load_patches(str(DATA), size=PATCH, stride=PATCH // 2, keep_empty=KEEP_EMPTY,
                          max_month_offset=MAX_MONTH_OFFSET, max_per_scene=MAX_PATCHES_PER_SCENE, dtype=np.float16)
metas = ps.metas
tr, va = dataset.split_patches(metas, PATCH, HOLDOUT)
lab = ps.Y[ps.Y != features.IGNORE]
print(f"{len(ps)} patches ({features.N_CHANNELS} channels, S1 part {ps.s1.shape[1:]}, {(ps.s1.nbytes + ps.Y.nbytes) / 1e6:.0f} MB) | train {len(tr)} | val {len(va)} | "
      f"water share of labelled pixels {np.mean(lab == 1):.3f}")
print("per site:", pd.Series([m['site'] for m in metas]).value_counts().to_dict())
print("per pass direction:", pd.Series([m['orbit'] for m in metas]).value_counts().to_dict())
assert len(tr) and len(va), "need both train and validation patches: check HOLDOUT / the manifest (a held-out site needs usable scenes)"

## 5. Train TerraMind

`TerraMindWater` = pretrained TerraMind encoder (`S1RTC`: terrain-corrected gamma0 in dB, standardised with TerraMind's own statistics; it reads VV and VH only) + multi-layer
token decoder + a pixel stem that sees **all 14 channels** (VV, VH, SDWI, relief, slope, aspect, range slope, land-cover groups): terrain and SDWI reach the decoder by late fusion.
The encoder is frozen by default. Loss = cross-entropy (ignored pixels excluded) + Dice. Land-cover channels are zeroed in half of the samples. There is no flip or rotation
augmentation, because aspect and the radar-look slope would stop matching the image. The best epoch by validation IoU is kept, and the checkpoint is written to Drive.

In [ ]:
tm.torch.manual_seed(0); np.random.seed(0)
model = tm.TerraMindWater(BACKBONE, pretrained=PRETRAINED, unfreeze_last=UNFREEZE_LAST)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{BACKBONE}: {n_train/1e6:.2f}M trainable of {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")
state, hist = tm.train(model, ps, tr, va, epochs=EPOCHS, batch=BATCH, lr=LR, device=DEVICE)

h = pd.DataFrame(hist)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(h.epoch + 1, h.loss, marker="o"); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(h.epoch + 1, h.val_iou, marker="o", color="C1"); ax[1].set_title("validation IoU (water)"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

# two files: the complete model (pretrained encoder inside: large, keep on Drive) and only the weights training changed (a few MB: fine for GitHub)
CKPT = MODEL_DIR / "terramind_water.pt"
CKPT_LIGHT = MODEL_DIR / "terramind_water_decoder.pt"
best_iou = float(h.val_iou.max()) if h.val_iou.notna().any() else None
TRAIN_META = {"backbone": BACKBONE, "pretrained_encoder": PRETRAINED, "unfreeze_last": UNFREEZE_LAST, "sites": SITE_IDS, "holdout": [sites.slug(h) for h in HOLDOUT],
              "labels": "JRC monthly (same month), terrain-corrected", "channels": list(features.CHANNELS), "orbits": sorted({m["orbit"] for m in metas}), "start": START, "end": END, "patch": PATCH, "epochs": EPOCHS, "lr": LR,
              "n_patches": len(ps), "n_train": len(tr), "n_val": len(va), "best_val_iou_water": best_iou,
              "trained_at": pd.Timestamp.now().isoformat(timespec="seconds")}
tm.save_checkpoint(model, str(CKPT), TRAIN_META)
tm.save_checkpoint(model, str(CKPT_LIGHT), TRAIN_META, trainable_only=True)
(MODEL_DIR / "training_history.json").write_text(json.dumps({"meta": TRAIN_META, "history": hist}, indent=2, default=str))
for p_ in (CKPT, CKPT_LIGHT, MODEL_DIR / "training_history.json"):
    print(f"saved {p_}  ({p_.stat().st_size / 1e6:.1f} MB)")

### Check the saved models

Reload both files and compare their output with the model that was just trained: the difference should be ~0. The small file rebuilds the pretrained encoder from
Hugging Face and puts the trained weights on top, so this also proves that file is complete.

In [ ]:
xb = tm.torch.from_numpy(ps.batch(va[:4])[0]).float().to(DEVICE)
model.eval()
with tm.torch.no_grad():
    ref = model(xb)
for p_ in (CKPT, CKPT_LIGHT):
    m_, _ = tm.load_checkpoint(str(p_), device=DEVICE)
    with tm.torch.no_grad():
        diff = (m_(xb) - ref).abs().max().item()
    print(f"{p_.name}: max |logit difference| vs the trained model = {diff:.2e}")
    assert diff < 1e-3, f"{p_.name} does not reproduce the trained model"
del m_

### Validation per site

IoU is measured against the **training labels** (JRC with the terrain corrections, on the labelled pixels only), so it shows how well the model reproduces them on a site it has not
seen. It is not accuracy against reality: for that, compare with an independent reference (a clear Sentinel-2 date, OPERA DSWx, or hand-checked patches).

In [ ]:
VAL = {}
for s in sorted({metas[i]["site"] for i in va}):
    idx = [i for i in va if metas[i]["site"] == s]
    VAL[s] = tm.evaluate(model, ps, idx, device=DEVICE)
    print(f"{s:>8}: IoU(water) vs labels = {VAL[s]:.4f}  ({len(idx)} patches)")

### Keep the small model in the repository

Copies the small checkpoint to `models/` in the repository together with a model card (`.json`): what it was trained on, how it did on the held-out sites, and how
to load it. `models/*.pt` is allowed by `.gitignore` (the large full checkpoint in `outputs/` is not), so this is what you commit and push. GitHub rejects files above
100 MB; if the file is bigger than that (a very large backbone or many unfrozen blocks), use Git LFS or a GitHub Release instead.

In [ ]:
import shutil
REPO_MODELS = ROOT / "models"
REPO_MODELS.mkdir(exist_ok=True)
shutil.copy(CKPT_LIGHT, REPO_MODELS / "terramind_water_decoder.pt")
size_mb = (REPO_MODELS / "terramind_water_decoder.pt").stat().st_size / 1e6
card = {**TRAIN_META, "val_iou_water_vs_labels_per_site": {k: round(float(v), 4) for k, v in VAL.items()}, "file_mb": round(size_mb, 1),
        "note": "IoU is against the JRC (terrain-corrected) training labels, not against ground truth.",
        "load": "from ai_water import terramind_seg as tm; model, meta = tm.load_checkpoint('models/terramind_water_decoder.pt')  # needs Hugging Face for the pretrained encoder"}
(REPO_MODELS / "terramind_water_decoder.json").write_text(json.dumps(card, indent=2))
print(f"models/terramind_water_decoder.pt  {size_mb:.1f} MB  |  card: models/terramind_water_decoder.json")
if size_mb > 90:
    print("WARNING: above ~90 MB: GitHub rejects files over 100 MB. Use Git LFS or a GitHub Release for this file.")
print("To publish:  git add models/ && git commit -m 'Add trained TerraMind water model' && git push")

## Next

`models/terramind_water_decoder.pt` (and its model card) is the model to commit. Open **`ai_water_inference.ipynb`** to map water with it: Angat Dam by default,
another dam by supplying its vector file, many bodies at once, or the whole country.

- **Labels are not ground truth.** They are JRC (Landsat, monthly, 30 m) matched to the Sentinel-1 month and corrected with the terrain. The IoU above is against those labels:
  check the model against an independent reference (a clear Sentinel-2 date, OPERA DSWx, or hand-checked patches) before relying on it.
- **Look at the hard negatives.** The `hard_neg_km2` column of the table above is the dark-but-not-water land the model is trained to reject. A site with none teaches nothing about
  shadow: if the reservoirs you map are in steep terrain, train on steep sites (Angat is one).
- **Check the contrast column.** Low contrast on a scene usually means the JRC month does not match the Sentinel-1 date (the reservoir moved within the month).
- **Rivers a few pixels wide** are mostly mixed pixels at 10 m: expect them to be the weakest of the six. The held-out site tells you how well the model carries over.
- An interrupted dataset build resumes (section 3). To add water bodies, add rows to `data/training_sites.csv`.